**Landasan Teori & Fakta Ilmiah (Bahan Slide Presentasi Besok)**

Sebelum masuk ke kodingan, berikut dasar teorinya:

* **Metode Benjamin Graham Number:** * *Sumber Teori:* Buku *"The Intelligent Investor"* oleh Benjamin Graham (Guru dari Warren Buffett).
* *Fakta Matematis:* Graham menetapkan bahwa rasio P/E (Price to Earnings) dikali P/B (Price to Book Value) suatu saham idealnya tidak boleh melebihi angka **22.5** ($15 \times 1.5$).

**Formula:**
$$\text{Harga Wajar (Graham Number)} = \sqrt{22.5 \times \text{Current EPS} \times \text{Book Value Per Share}}$$




* **Piotroski F-Score:**
* *Sumber Teori:* Riset Joseph Piotroski (2000), *"Value Investing: The Use of Historical Financial Statement Information to Separate Winners from Losers"*.
* *Fakta Matematis:* Menggunakan skala ordinal 0–9 untuk mengukur kekuatan finansial riil perusahaan dari aspek profitabilitas, *leverage*, dan efisiensi operasi.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [3]:
print("[*] Menginisialisasi Pipa Data Fundamental Eksekutif...")

# 1. LOAD DATA MASTER STATISTIK (Harta Karun Lu)
# Pastikan nama file CSV sesuai dengan yang ada di environment Kaggle lu
df_stats = pd.read_excel('../data/IDX Fundamental Analysis 2026-04-04.xlsx', sheet_name='key-statistics')

# Bersihkan spasi gaib di nama kolom biar ga kena KeyError
df_stats.columns = df_stats.columns.str.strip()

# FILTER TARGET: Hanya emiten sektor emas acuan Capstone lu
target_emiten = ['ANTM', 'BRMS', 'MDKA', 'PSAB', 'HRTA']
df_gold = df_stats[df_stats['Ticker'].isin(target_emiten)].copy()

[*] Menginisialisasi Pipa Data Fundamental Eksekutif...


In [4]:
print("[*] Menghitung Nilai Intrinsik via Rumus Benjamin Graham...")

# Ambil parameter inti dari kolom excel lu
eps = df_gold['Current EPS (TTM)'].fillna(0)
bvps = df_gold['Current Book Value Per Share'].fillna(0)

# Jaring Pengaman: Nilai perkalian ga boleh minus (< 0) biar ga menghasilkan nilai imajiner pas di-akar
perkalian_graham = np.clip(22.5 * eps * bvps, 0, None)
df_gold['Harga_Wajar_Graham'] = np.sqrt(perkalian_graham)

[*] Menghitung Nilai Intrinsik via Rumus Benjamin Graham...


In [5]:
print("[*] Melakukan Kalibrasi Skor Kesehatan Piotroski...")

# Ambil skor asli (0-9) dan konversi ke rentang Fuzzy (-1 sampai +1)
if 'Piotroski F-Score' in df_gold.columns:
    scaler_fuz = MinMaxScaler(feature_range=(-1, 1))
    df_gold['Skor_Piotroski_Fuzzy'] = scaler_fuz.fit_transform(df_gold[['Piotroski F-Score']].fillna(0))
else:
    # Fallback plan jika kolom tidak terbaca sempurna
    df_gold['Skor_Piotroski_Fuzzy'] = 0.0

[*] Melakukan Kalibrasi Skor Kesehatan Piotroski...


In [6]:
df_output_fundamental = df_gold[[
    'Ticker', 
    'Current EPS (TTM)', 
    'Current Book Value Per Share', 
    'Harga_Wajar_Graham', 
    'Piotroski F-Score', 
    'Skor_Piotroski_Fuzzy'
]]

df_output_fundamental.to_csv('fundamental_evaluasi_final.csv', index=False)
print("[+] Sukses mengekspor 'fundamental_evaluasi_final.csv'!")

[+] Sukses mengekspor 'fundamental_evaluasi_final.csv'!


In [8]:
print("\n📊 TABEL EVALUASI FUNDAMENTAL EMITEN EMAS:")
df_output_fundamental


📊 TABEL EVALUASI FUNDAMENTAL EMITEN EMAS:


,Ticker,Current EPS (TTM),Current Book Value Per Share,Harga_Wajar_Graham,Piotroski F-Score,Skor_Piotroski_Fuzzy
0,ANTM,299.98,1468.88,3148.694810,8,1.0
1,BRMS,5.81,147.52,138.868830,8,1.0
2,MDKA,-41.79,540.73,0.000000,6,-1.0
3,PSAB,18.89,207.88,297.244339,7,0.0
